#  1— Dataset Preparation & Analysis

**SparkAI Computer Vision Engineer Assessment — MVTec AD Bottle**

Binary classification:
- `normal` = defect-free bottle
- `defective` = bottle with a defect

**Important:** MVTec AD is officially an anomaly-detection benchmark. This notebook creates a custom binary-classification protocol for this assessment.


In [ ]:
from pathlib import Path
from collections import Counter
import random, shutil
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ROOT = Path.cwd()
RAW = ROOT/"data"/"raw"/"mvtec_ad"/"bottle"
OUT = ROOT/"data"

if not RAW.exists():
    raise FileNotFoundError(f"Expected: {RAW}")

print("Dataset path:", RAW)


In [ ]:
EXTS={".jpg",".jpeg",".png",".bmp",".webp"}
def imgs(folder):
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in EXTS)

train_good=imgs(RAW/"train"/"good")
test_good=imgs(RAW/"test"/"good")
defect_dirs=[p for p in (RAW/"test").iterdir() if p.is_dir() and p.name!="good"]
defective=[p for d in defect_dirs for p in imgs(d)]

print("Train good:",len(train_good))
print("Test good:",len(test_good))
print("Defect groups:",[d.name for d in defect_dirs])
print("Defective:",len(defective))


In [ ]:
sizes=Counter(); bad=[]
for p in train_good+test_good+defective:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im: sizes[im.size]+=1
    except Exception as e:
        bad.append((str(p),str(e)))
print("Image sizes:",sizes.most_common())
print("Unreadable images:",len(bad))


In [ ]:
records=(
    [{"path":str(p),"label":"normal","label_id":1} for p in train_good+test_good]
    + [{"path":str(p),"label":"defective","label_id":0} for p in defective]
)
df=pd.DataFrame(records).sample(frac=1,random_state=SEED).reset_index(drop=True)

print(df["label"].value_counts())
df["label"].value_counts().plot(kind="bar",title="Class Distribution")
plt.ylabel("Images"); plt.tight_layout(); plt.show()


In [ ]:
train_df,temp=train_test_split(df,test_size=.30,stratify=df.label,random_state=SEED)
val_df,test_df=train_test_split(temp,test_size=.50,stratify=temp.label,random_state=SEED)

for name,d in [("train",train_df),("validation",val_df),("test",test_df)]:
    print(name, len(d), d.label.value_counts().to_dict())


In [ ]:
# Create ImageFolder-compatible directories.
for split,d in [("train",train_df),("val",val_df),("test",test_df)]:
    for label in ["normal","defective"]:
        folder=OUT/split/label
        folder.mkdir(parents=True,exist_ok=True)
        for old in folder.iterdir():
            if old.is_file(): old.unlink()
    for i,row in d.reset_index(drop=True).iterrows():
        src=Path(row.path)
        dst=OUT/split/row.label/f"{i:05d}_{src.parent.name}_{src.name}"
        shutil.copy2(src,dst)

for split in ["train","val","test"]:
    print(split,{c:len(list((OUT/split/c).glob("*"))) for c in ["normal","defective"]})
